# 植物幼苗分类：DINOv2-B 第 8 层微调 + SVC（Kaggle）

这是项目最新推荐方案的独立 Notebook：**种子 142、原五折、336×336 输入、
分类头预热 2 轮、骨干第 7/8 个 block 微调 6 轮、训练折内三折选择 SVC 参数**。
五折的测试概率等权平均，最后生成 `ID,Category` 格式的 `submission.csv`。

本地历史 OOF 为 **96.20% Accuracy / 96.20% Macro-F1**。不同 GPU、PyTorch
版本和浮点计算可能带来复现波动；实际成绩以本次运行结果为准。测试图没有标签。

**使用顺序**

1. 在 Kaggle 导入本 `.ipynb`，通过 **Add Input** 添加任务数据。
   可直接上传配套 `task2_dataset_for_kaggle.zip` 创建 Dataset 并添加。
2. 在 **Settings → Accelerator** 选择 GPU，并开启 **Internet** 供首次下载权重。
3. 按默认配置 **Run All**。结果会显示 OOF 指标，并导出
   `/kaggle/working/submission.csv`。

数据目录应包含 `train/<类别名>/*.png` 和 `test/*.png`。默认自动定位；
有多个符合条件的数据目录时，在下面设置 `DATA_ROOT`。
代码、原折划分和原提交 ID 顺序已内嵌，无需另外上传 `.py` 文件。

[Kaggle Notebook 官方说明](https://www.kaggle.com/docs/notebooks) ·
[DINOv2 官方项目](https://github.com/facebookresearch/dinov2)

In [ ]:
from pathlib import Path

# 通常只需添加数据，保留下面的默认值即可。
DATA_ROOT = ""                  # 可选，例如 /kaggle/input/my-task2/dataset-for-task2
SUBMISSION_TEMPLATE = ""        # 可选；留空使用内嵌的原始 378 个 ID 顺序
LOCAL_PRETRAINED = ""           # 离线时指定官方 .pth；也会在 /kaggle/input 自动寻找同名权重
PREFERRED_DATASET_SLUG = 'plant-seedlings-task2-20261007-8d15d7'
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/plant_midlayer")
SUBMISSION_OUTPUT = Path("/kaggle/working/submission.csv")

SEED = 142
IMAGE_SIZE = 336
BATCH_SIZE = 4
ACCUMULATION = 4                # 有效 batch = 16
WORKERS = 0
WARMUP_EPOCHS = 2
FINETUNE_EPOCHS = 6
FOLDS = [0, 1, 2, 3, 4]         # 调试可用 [0]；完整提交需要五折全部完成
RUN_NAME = "dinov2b_mid_finetune_seed142"
RESUME = True                  # 同一配置重跑时跳过已经完成的折
VERIFY_RELOADED = False         # True 会重新加载五折权重，再推理测试集核验
INSTALL_MISSING_PACKAGES = True
SVC_PACKAGE_VERSION = "1.7.2"  # 与原实验一致；离线环境已安装时可设置为 None

LR_WARMUP = 1e-3
LR_BACKBONE = 1e-5
LR_HEAD = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
C_GRID = "0.1,1,10,100,1000"

if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ("", ".", ".."):
    raise ValueError("RUN_NAME 必须是单个目录名称")
RUN_DIR = WORK_DIR / "experiments" / RUN_NAME

## 1. 依赖与 GPU

使用 Kaggle 自带的 PyTorch/CUDA，只补充缺失的常规依赖，并将 scikit-learn
设为原实验版本。模型使用 PyTorch 的注意力实现，与本地实验一致。

In [ ]:
import os
import sys
import subprocess
import importlib.util
from importlib import metadata

if sys.version_info < (3, 10):
    raise RuntimeError("需要 Python 3.10 或更新版本")
os.environ["XFORMERS_DISABLED"] = "1"

dependencies = {
    "numpy": "numpy>=1.26,<3",
    "sklearn": "scikit-learn" + (f"=={SVC_PACKAGE_VERSION}" if SVC_PACKAGE_VERSION else ">=1.3"),
    "joblib": "joblib>=1.3,<2",
    "PIL": "Pillow>=10,<13",
}
missing = [requirement for module, requirement in dependencies.items()
           if importlib.util.find_spec(module) is None]
if SVC_PACKAGE_VERSION and importlib.util.find_spec("sklearn") is not None:
    if metadata.version("scikit-learn") != SVC_PACKAGE_VERSION:
        missing.append(dependencies["sklearn"])
if missing:
    if not INSTALL_MISSING_PACKAGES:
        raise RuntimeError(f"请安装依赖或开启 INSTALL_MISSING_PACKAGES：{missing}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                               "--upgrade-strategy", "only-if-needed", *missing])
    except subprocess.CalledProcessError as error:
        raise RuntimeError("依赖安装失败；请开启 Kaggle Internet 后重跑此单元格") from error

import base64
import csv
import hashlib
import io
import json
import random
import shutil
import time
import zipfile
from collections import Counter
from dataclasses import dataclass
from types import SimpleNamespace
from typing import Sequence

import joblib
import numpy as np
import sklearn
import torch
import torchvision
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from IPython.display import FileLink, display

if not torch.cuda.is_available():
    raise RuntimeError("当前没有 CUDA GPU。请在 Settings → Accelerator 启用 GPU 后重新运行。")
if not hasattr(torch.amp, "GradScaler"):
    raise RuntimeError("需要支持 torch.amp.GradScaler 的 PyTorch 版本")
WORK_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(WORK_DIR / "torch_hub"))
environment = {
    "python": sys.version.split()[0], "torch": str(torch.__version__),
    "torchvision": torchvision.__version__, "numpy": np.__version__,
    "scikit_learn": sklearn.__version__, "joblib": joblib.__version__,
    "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
}
(WORK_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print(json.dumps(environment, indent=2))
print("本方案使用单个 GPU。")

## 2. 原实验资源

下方折叠的单元格含原 500 张训练图的五折划分、378 个测试 ID，以及本地实验
所用的 DINOv2 骨干源码。官方文件保持原内容，并附 Apache 2.0 许可证。
权重不嵌入 Notebook；首次运行下载约 330 MiB 的官方预训练权重并核对 SHA-256。

In [ ]:
# 内嵌资源；无需手动修改。
EXPECTED_CLASSES = ['Black-grass', 'Common wheat', 'Loose Silky-bent', 'Scentless Mayweed', 'Sugar beet']
SOURCE_MANIFEST = {'reference': 'DINOv2-B layer 8 partial fine-tuning, seed 142', 'split_sha256': 'd0514b5c6a0c508f8d16ed8fd00bf5d14ca2e7d244d0e6253ac20e7e36f3b772', 'pretrained_sha256': '73182a088cf94833c94b1666d1c99e02fe87e2007bff57b564fb6206e25dba71', 'dino_archive_sha256': 'b114c3dcece205f72b55c4816fd7963f76e30706da79ebcafa6778adfdcbf342', 'dino_source_sha256': {'LICENSE': '600cc67cc4cb2f5ea317dcfc687ad1c74dc4bec8782bbe9db0afd83513b935b7', 'dinov2/__init__.py': '0d2b87b7e71c7f7279ccc4b4f8b541d26071425284a0ea01a5a435fac4511cfe', 'dinov2/hub/__init__.py': '2932e94a2d67be87b717c49468214c1127be7864aedf37f5891d26be7bc1437f', 'dinov2/hub/backbones.py': '871fca671b12a9ff02e810654baf509e97ccf461bf8196ce5ddeefff2fd87d3e', 'dinov2/hub/utils.py': '579613e3d7b82c2a387eddc215ad9f076b8740f9618229628395fe5755dbd131', 'dinov2/models/__init__.py': 'b93329ce676f3eb0c9a7790d1b3b2b114a0855ea8dc76e32e262d1a7dc6c55ae', 'dinov2/models/vision_transformer.py': '7799a260f2d7d0fe197331d08502fb8c542f9b7424723650f6a39b64fa2639ea', 'dinov2/layers/__init__.py': '1b55deed39d5ab0b589bef421bbfe06f24a10cd590a0f8403234c9ee4e34109d', 'dinov2/layers/attention.py': '79c7be7a452b3aad96698ec38d5d5150b9f4d8ac084fa93324510dc9f624775d', 'dinov2/layers/block.py': '60c0ac7dfa4474be313fabfa5a23d82faf6f0cecd4e720a88be35de9788cb636', 'dinov2/layers/dino_head.py': '9fdb1fa61c0609dc0876f711f9d0d828c33bf9fd972aab3d9c968d44e16b98d1', 'dinov2/layers/drop_path.py': 'b9f8236e86054b9d9a71275efcad2a9ecaa1f86b529d4b8d6109ddb5e806f67a', 'dinov2/layers/layer_scale.py': 'dadd5aafe178f1bf72a205a02a6645c7e635cacbad585d4a7369c200c6e89135', 'dinov2/layers/mlp.py': '255825c73b60a916dd00eb1e38aacbcdbf316e40d6a005efb46e245b7edb43aa', 'dinov2/layers/patch_embed.py': '40da6add3d811198ea3e17cb99cdd4e5cda59e369efbbe3d18d89308618cf142', 'dinov2/layers/swiglu_ffn.py': 'e46d2948fb97e497cf991ff82ce30cb59b59fdf2e12a19568417113716b4f119'}, 'experiment_source_sha256': {'train_classifier.py': '1cdcdf675dca05d99e81a1e93b59af1f9ddc4bcd45a243529a5031a3b6c704c9', 'train_dino_token.py': 'd6815da4f6a0478c03dcc16c7bf1a38890c592f34afb0282985d1d814e53db23', 'joint_foreground_modules.py': '73045bc956d9dd365f256fcbdff343a543fb37a06271836a83025d53c9a38065', 'midlayer_finetune_framework.py': 'ba573400b468ad0a1a9e9566963baf387cef4675e25ac33837c8b90ec762806a', 'evaluate_frozen_patch_probe.py': '137dd56d8355cec53d0e464d585d415f5f902c8da7159e854399b840afdad874', 'train_midlayer_finetune.py': 'fd4b8e161025450bc240863fc0c62bba1528bb211381d022544e5be07c2376b6', 'predict_midlayer_finetune.py': 'b3b39031199ff2bc039c5db21e03dda8831bb408bbce9e63215794144dac87d3'}}
FOLDS_BASE64 = """
...
cGFjdGEucG5nLFN1Z2FyIGJlZXQsNCw0DQo=
"""
TEMPLATE_BASE64 = """
SUQsQ2F0ZWdvcnkKeGd1d3piYnp3ay5wbmcsCm9zd2trb3dxdnoucG5nLApxY3JrYWVzbHZsLnBuZywKZHRkc2xiZWZ5eC5wbmcs
CnlpaWxleWpqaGUucG5nLApubGx1ZG95a2xhLnBuZywKY25oZWFxaHhxdC5wbmcsCnJ6emprbmFwanMucG5nLApncnpkcWhnZWFr
LnBuZywKdXZ0dm1ibHZqby5wbmcsCmpjaWx6cGliYnoucG5nLAp5aWJuYWZvYXhmLnBuZywKc3BkZ2RrY2VwcS5wbmcsCmVqdGx3
aXJoZXIucG5nLApkZmdianFuYXRpLnBuZywKZ25wc3h6bXR5Yy5wbmcsCmtscG5wanlpdnAucG5nLAp5eGlzYnF5c2NtLnBuZywK
dXNzb3VvbmpnZC5wbmcsCnh1b2txa2h6dG4ucG5nLApjY3NuZGZha29hLnBuZywKbHFleG96Y3doby5wbmcsCnRuYnZpcnhzd2Qu
cG5nLApmdnp0Y3J1Z3F1LnBuZywKd2V0eG1zbnJ6dy5wbmcsCnhzemp6ZHN3cGIucG5nLAp1cXhyeGh6c3drLnBuZywKbmZvYmRz
aGZndy5wbmcsCnZ5b3RsZW9qZmQucG5nLAp1b3h2bHdiZHd4LnBuZywKb3BjZW96ZmtqdS5wbmcsCmpndXhvYmJqbXQucG5nLApx
b2tsanJ6eXdwLnBuZywKaGN6b2J6c2VkdC5wbmcsCmt4am1ibWtlbG4ucG5nLApsdXhld3lxY3NoLnBuZywKZmNreWZud3J5cy5w
bmcsCmVuZXNsaWNyc2YucG5nLApmZmpscGh4aXV1LnBuZywKZnlmdWJnZHVxdC5wbmcsCnZ0a2Rib2xhdG4ucG5nLApvd3lzY2hr
cWJlLnBuZywKcG9tdWN2aGxmdi5wbmcsCndqYXd6eWZvb3kucG5nLApmdmFtem1saWVxLnBuZywKdGV4YWRxeGZ0ay5wbmcsCm50
enlzZGtkaXgucG5nLApud3JreXdlaGN5LnBuZywKcGVkdWJ4cGdkdC5wbmcsCnhjdm9pd3F2Y2cucG5nLApiZWhyeHd6dWViLnBu
ZywKeWVwZHphaWFpYi5wbmcsCm93Yml6enR4aW8ucG5nLApucm5yaXd1ZWpkLnBuZywKamtxbmd2bGV3di5wbmcsCnFhYnJkemN6
cXYucG5nLApraWtnZ2VqY3p3LnBuZywKYmZndHBoaG1pdS5wbmcsCmt1ZHNlbmV4ankucG5nLAp6YWx4ZmNzcHV3LnBuZywKZW9v
bm1ibG15cy5wbmcsCmtna3JzaWFxdnAucG5nLApiZGR5dmR2a2dvLnBuZywKdnpkdW1vZG1peC5wbmcsCm12a2RzdmlvZm8ucG5n
LAp3d3N3a2JyY29xLnBuZywKcXRqZGRzYm9kei5wbmcsCnhvbmtwbHRva2MucG5nLApsdGV6em16YWxpLnBuZywKeWhmaWhvcm9q
YS5wbmcsCmJpd3Bqd3l0ZHoucG5nLAplaXZ5cXVraGJtLnBuZywKcnZmbW14aHdpei5wbmcsCmtxd3R2emFvamgucG5nLApjb3B6
d3JjcXBwLnBuZywKZ3hubHRudXNydy5wbmcsCnRrenR4bmJ6aWkucG5nLAprbGF4dHh3enB5LnBuZywKaWJ3aWFoeHpqZC5wbmcs
CmJqb2dqdnZnb28ucG5nLAppa3F4a2dxY29lLnBuZywKdHl3YWJzdnFyYS5wbmcsCnhvbW54Y3l0dnUucG5nLApneWRoZ2ptYmV1
LnBuZywKcHZoY2dwd3htaC5wbmcsCmdxYXF4ZnNyam0ucG5nLApxdGlqZHZob2xnLnBuZywKa3ZkZHdweWRvZi5wbmcsCnZmcWp4
a3RxbGEucG5nLApiYnRhYm9xa3l5LnBuZywKd25yaWxldGx1Yi5wbmcsCmFjbWJnamZsY3QucG5nLApheGNrenFhdHR4LnBuZywK
dnBibWhha2dody5wbmcsCm1mb214b2pvcHQucG5nLApiY2Z0dnppd29jLnBuZywKZGh1eGdmY2J5dy5wbmcsCmNlbnR1cnJ4YXMu
cG5nLApieWFnY2plenFzLnBuZywKZWdwemR4cXZwZC5wbmcsCnJuaGdoYWJuamEucG5nLAp6b2hkdml2YmFtLnBuZywKb2N6bGti
enpleC5wbmcsCnp5YW5ubWpuenAucG5nLAphc2hjdHh1eWZmLnBuZywKb3R4c2ltb2dhcy5wbmcsCmhkeWlxanVqaHUucG5nLApw
dWdkcHFrcGRzLnBuZywKZnZlb21qZHp1by5wbmcsCnNyaWFlc2h5ZmsucG5nLAprd3NrZG5qZ3hjLnBuZywKYWlqbWRhb2N5bS5w
bmcsCnlyd2ptZmh3cnkucG5nLApib2RrdHBvZnVpLnBuZywKenV3aGpoYWN3cy5wbmcsCm10d3ByZm5nc20ucG5nLApkY3VqZG12
c2ljLnBuZywKcG16YXNmeWhjZi5wbmcsCmZ1cXloZHBkdGQucG5nLApkYW50YXJlc3F3LnBuZywKcnlzamt3aWxlYS5wbmcsCnJv
emprbnRhbGYucG5nLApzemt2eGVjemd4LnBuZywKbXNsb3ZlemFzdC5wbmcsCnJqbnFmaGtrZ2cucG5nLAphemxnbnV3emNjLnBu
ZywKZGtrdWRpYnNjdi5wbmcsCmFibHprcG9qbGEucG5nLApscnl4ZG94eHd0LnBuZywKbHd2ZndidHZyaC5wbmcsCmt6aW1taGdr
cHoucG5nLApqdXd0YnFoc3lvLnBuZywKbnBwd2p5am1pai5wbmcsCmprZ2tneW5la2cucG5nLAplc3FpaWthZmtnLnBuZywKbHp4
cGVpb2hrZC5wbmcsCmx4Z3pza3Bud3gucG5nLApsdXFpb2hueGNlLnBuZywKYmNneGd5YmN0Yi5wbmcsCnl0c3VsdmJ6eHYucG5n
LApmb25oZG51ZWlwLnBuZywKYWZoY255dGp0ZC5wbmcsCnR3b2l6a3prdW8ucG5nLAp3ZXZ3bWVkdm9iLnBuZywKeXV6dHBqdG1l
ZC5wbmcsCnNtdXhwc25tZnUucG5nLApoYnhneWZ0eXF2LnBuZywKcGZmYnJ3bndway5wbmcsCmNtZnhsYm5yYnAucG5nLAp2eW5y
eW5rd2ttLnBuZywKZ2h5cHdkYmNmZi5wbmcsCmlyamJzaXF0Z3MucG5nLAptY2p5ZGZ6b2x6LnBuZywKcm9wa2RoenJ6bS5wbmcs
CmxocHZxdHRqaWUucG5nLApybGlnZWZ0ZXVnLnBuZywKdHhpd3ZsYWtndS5wbmcsCmRtYXZ2dHFveHQucG5nLAp1cndlbXp1c2Rn
LnBuZywKZ29waHFjdWxrby5wbmcsCnRnYnpxb2RobHgucG5nLAppeXh4enZubHhoLnBuZywKZHdwdm5haW9rdi5wbmcsCnRtYmh5
a3lramcucG5nLAp3eGp4bWZmZWVkLnBuZywKZHJ6ZHF6cGd3ZS5wbmcsCm9uYXZsd3Vya3EucG5nLAprZW5jY3JibHJlLnBuZywK
cGhnd3B3dnFrYi5wbmcsCm9zbnVpemJ2am4ucG5nLAp0bWtqZnh4dmhuLnBuZywKYWhveWV5dm9zZy5wbmcsCmdsc2d2cW9lc3ku
cG5nLAp6dWpjdG9sY3FjLnBuZywKd2J4a3V4dmxlbi5wbmcsCnF0d3ZtcHdxeHIucG5nLApwam9iZHd1cmhxLnBuZywKZWtoZ2Nv
cGlpcS5wbmcsCnFlcHJnZmxqYXoucG5nLAp2emd3d3J1cWRiLnBuZywKdWp1bHpiZWt3bS5wbmcsCnh2YXpjcHpqbGUucG5nLApy
Z2J0YnpreGtmLnBuZywKcnlna2NkYnJydi5wbmcsCm53cndhd3RyenYucG5nLAppYWR1aW5nY3ZwLnBuZywKdm5odGl5YXdwdS5w
bmcsCmZwZWptbmVkdnIucG5nLAp1dWFoZmhndm13LnBuZywKYXJtcHdjdGppcC5wbmcsCmR0YWJrZmVtYXoucG5nLApqYWhjZGRq
dnlyLnBuZywKYnZhZXBjY2R1dy5wbmcsCm1teW1vdXR6aWIucG5nLApvY3lscGtoZW12LnBuZywKaHdidnZ6a3V0Zi5wbmcsCmZ1
eWV6dnlzdm4ucG5nLAp0aHFleXNwc3R2LnBuZywKc2x0Ym9pdGFkcC5wbmcsCnN5eHpvcGV4aXAucG5nLAp2bGhwbXNxY3F6LnBu
ZywKb2pmdWNleXVzdS5wbmcsCnRjc2doc2lid3YucG5nLApscmJ5c3N0b21tLnBuZywKenplb2VmaHpsbS5wbmcsCmFwanJuYXZy
bnIucG5nLAprZGN6eXV1aXhiLnBuZywKbXp4eWt6eHhsZS5wbmcsCmpuZHN1anZoamUucG5nLAp6aGprcG11aWtjLnBuZywKaHlx
cGhjYmVydS5wbmcsCmthcXV4bW92dnAucG5nLAprbnVtaGNhYXByLnBuZywKbnhlam1oamdvaC5wbmcsCmNndWpwdHVhZWMucG5n
LApqYmphdXp4dHhiLnBuZywKaGt4dHlpcHp4ei5wbmcsCnZicml0cXFqb3cucG5nLAp4ZnRtem5tdHh2LnBuZywKY3Z6ampndXNx
cC5wbmcsCmV3eWN0dWNmY2sucG5nLAp6bGh6b2JlaGNsLnBuZywKYnB0bmFjdXNnZi5wbmcsCnBkZWFpYmh4aXUucG5nLApmaGx0
c2xtZnprLnBuZywKY3pqeWNvdWN1aC5wbmcsCmp2YXlzaHdxZ2QucG5nLAp4bGh3amNiYmV4LnBuZywKeHdneHNkcmF5ei5wbmcs
CnpuYmZ2aG5mam0ucG5nLApsZGt3bWdvcWljLnBuZywKYnhyZ3Zscnpmdi5wbmcsCm10aXZyY3V4ZXgucG5nLAp3eHd1d3VqY3Rv
LnBuZywKYWJraWV1cmZyci5wbmcsCm1yeGd0aGNvbmMucG5nLApveHFha3dlbmRnLnBuZywKdGhyanpldGNrby5wbmcsCmtxbHhh
a2Jmd3AucG5nLAprcnJpcXh0dWR0LnBuZywKa3B2bWFyZWZpaC5wbmcsCnlkY3Vyd3hxc2YucG5nLApkbmJpYmZneG9oLnBuZywK
bW9peHdjZXd1YS5wbmcsCnp0cnBocnFoeHkucG5nLAp1aWl2b3NzbW9yLnBuZywKeHVzamdiY214cC5wbmcsCmNzbHd1cmJvcHou
cG5nLApqb3ZtYXZ2a2JhLnBuZywKaGlhYWptcG5veC5wbmcsCmpqb3dodWxhdmIucG5nLAp6dXhibmNpZnNnLnBuZywKZWFmY2Nj
amJpdi5wbmcsCmx5Y3Ric2JjYmwucG5nLAp1bHptanJtZWp5LnBuZywKcGp0cXptd2ZyYy5wbmcsCmRucWN1d2RhYXUucG5nLApq
aGxneHdhb2ZmLnBuZywKZm10YWNtaGdnZi5wbmcsCnhkc2duZ2xkcnYucG5nLApxa293bWdoaGVrLnBuZywKaWRmdHNibHZicS5w
bmcsCnF3dWxldGl4eXMucG5nLApsenl5a2ViZ2RkLnBuZywKYXVwZGxwY3pnZy5wbmcsCm5pd3NvdXh3ZmoucG5nLApsY2N6cGd6
cml6LnBuZywKbmViem9rYXVkbS5wbmcsCnJkbnJkaXJ1ZG4ucG5nLApzZHd5bmxycXpmLnBuZywKaWdkbmJ4Y2tsbC5wbmcsCnBh
ZmlrdnJkYXcucG5nLAp0amFnbGxnaXRuLnBuZywKZm5yaGJ4ampiby5wbmcsCmlwbXZwbWtneHIucG5nLAp2bW91dG5kbGF0LnBu
ZywKbnh3ZGZqY3VhZC5wbmcsCmZlZXRva2lwYnkucG5nLApkY2tmbWRjdm55LnBuZywKaHlqaXR1eXVpeS5wbmcsCnh5bWNod2dj
bGwucG5nLApnY2FjZG50dm1zLnBuZywKc3B4a3R5aWtobC5wbmcsCmdtd2p6am94cncucG5nLAp0eHRmdGJlanhyLnBuZywKdHNx
Z2NxdXphdC5wbmcsCm53aHd6enloZWkucG5nLApnY3BlZGlsbXR0LnBuZywKZmpiZnN6ZGFzcC5wbmcsCnh4ZmtkcnJra2oucG5n
LApzamlscWZidHVuLnBuZywKZnRrbXRkaW5pdC5wbmcsCnZ5bHBkdmlpZWMucG5nLAplY3R6aWZ0bmJjLnBuZywKaHdjcHF0YnJm
cS5wbmcsCmFqandyaGlwZ2gucG5nLApnamtyamJ5bWxwLnBuZywKZHJqb3F5enRueS5wbmcsCmFzZ2tkaGZsd3MucG5nLApranh4
aGdnZ2NtLnBuZywKYXJremNueGRkcS5wbmcsCmF4emp2cWFvenYucG5nLAp4aGlybmpwaHVnLnBuZywKbW1jcGt5bHRueS5wbmcs
Cnh4aWN5YXBrZGYucG5nLAptYWZibnNra2pvLnBuZywKcm9qaG5vcWhsbS5wbmcsCmNtZ25nbHpiZGkucG5nLApudGtucHZ2ZXVq
LnBuZywKdW9yd2VncWJlYy5wbmcsCmdjcXF2enFpY3QucG5nLApqd2ducWdyemloLnBuZywKZXFlcHNkZWltaC5wbmcsCnh4andq
bmtkdWcucG5nLApzdnp4dWJpcHF4LnBuZywKbXJsb3p2dGVnYS5wbmcsCm1obGRwbHFqaWYucG5nLApnbWxzdnNqeGV1LnBuZywK
Y3pyaGp1ZmF0dS5wbmcsCm94a3RkbWl2amcucG5nLApkeHd4em9rem5rLnBuZywKbWp3d2dmZXdwZi5wbmcsCnViYWZ2dWlrdWcu
cG5nLApjc3p2dHFjanlmLnBuZywKaXpxeWpjY3htcy5wbmcsCnBvc3B2ZGJrbHYucG5nLApncHB0d2Z3aHNxLnBuZywKc2pjdXFn
Y3ZjeC5wbmcsCmJ1Z21vbmlxcm8ucG5nLAplcGNua3JvdHRvLnBuZywKanhsaXB5dWxhdC5wbmcsCnNza210cW91dXEucG5nLApy
bnJnc2tqYmpiLnBuZywKbmxqb2Z5Zm95cS5wbmcsCmp4cndlc3lybHgucG5nLApzbGd4cXVwa2JvLnBuZywKbWRveWp5Ynd5Zy5w
bmcsCmZncHRzZWFiY2IucG5nLApvdXRlbmV2eHplLnBuZywKanVmcnRscmx2aS5wbmcsCnBjaXhueHl2ZXgucG5nLAp2YWFicWNz
ZGN5LnBuZywKeGNqb2tzc2Judy5wbmcsCnN0Y21pamhzaWgucG5nLApscHNyeW9maG1pLnBuZywKZmJlc21hZ2Nvdi5wbmcsCm95
d25ibGR0aG0ucG5nLApybGpjY2RuZ2hsLnBuZywKcmpnZmZvbWNvai5wbmcsCnByZWdvcW5mcWoucG5nLApza3Zzd2tsbXdyLnBu
ZywKbXR1YmpxbnNuai5wbmcsCnl3bGhhYW9zZ3kucG5nLAp0bmp2bHJjamdyLnBuZywKdHp5cGV0a2Jnbi5wbmcsCnJ2YWx2eXNj
cmQucG5nLApscHJ1Z2prd3plLnBuZywKbHluc3lpZW14ZS5wbmcsCm9qZG9jYnlucG4ucG5nLApoZmZuZGRsc3dtLnBuZywKd3ls
dXR1bmhtaS5wbmcsCnZwbmFiZ3FjbmwucG5nLApvZWFodXlraGpnLnBuZywKbWV0Z2tsdGh2cC5wbmcsCmxibW9vZXJ4eWgucG5n
LApuc2RubHNpZHdrLnBuZywKb210ZnlwcHhzZi5wbmcsCmdiY2xkdXN0eGEucG5nLApqY3Bmb3FncHdnLnBuZywKenl6bWpicHFo
ZS5wbmcsCmttZHZkampyamEucG5nLApubnF5cW50YWFiLnBuZywKZWh1a3BncHFjZy5wbmcsCnFoaGtwZmF1bG0ucG5nLAp1eWpl
dXFqbGhlLnBuZywKcG9sbWJnZ2txaC5wbmcsCnl5eXhrZ29heWMucG5nLApzbnNleG5iZGlrLnBuZywK
"""
DINO_SOURCE_BASE64 = """
...
"""

def extract_zip_safely(archive, destination):
    destination = Path(destination).resolve()
    for member in archive.infolist():
        target = (destination / member.filename).resolve()
        if not target.is_relative_to(destination) or "\\" in member.filename:
            raise ValueError(f"压缩包含无效路径：{member.filename}")
    archive.extractall(destination)

ASSET_DIR = WORK_DIR / "assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DIR = ASSET_DIR / "reference_split"
REFERENCE_DIR.mkdir(exist_ok=True)
SPLIT_PATH = REFERENCE_DIR / "fold_assignments.csv"
split_data = base64.b64decode(FOLDS_BASE64)
if hashlib.sha256(split_data).hexdigest() != SOURCE_MANIFEST["split_sha256"]:
    raise RuntimeError("内嵌折划分校验失败")
SPLIT_PATH.write_bytes(split_data)
EMBEDDED_TEMPLATE = ASSET_DIR / "sample_submission.csv"
EMBEDDED_TEMPLATE.write_bytes(base64.b64decode(TEMPLATE_BASE64))
with EMBEDDED_TEMPLATE.open(encoding="utf-8", newline="") as stream:
    ORIGINAL_TEST_IDS = [row["ID"] for row in csv.DictReader(stream)]

dino_data = base64.b64decode(DINO_SOURCE_BASE64)
if hashlib.sha256(dino_data).hexdigest() != SOURCE_MANIFEST["dino_archive_sha256"]:
    raise RuntimeError("内嵌 DINOv2 源码校验失败")
DINO_SOURCE_DIR = ASSET_DIR / "dinov2_source"
with zipfile.ZipFile(io.BytesIO(dino_data)) as archive:
    extract_zip_safely(archive, DINO_SOURCE_DIR)
for relative_name, expected_digest in SOURCE_MANIFEST["dino_source_sha256"].items():
    if hashlib.sha256((DINO_SOURCE_DIR / relative_name).read_bytes()).hexdigest() != expected_digest:
        raise RuntimeError(f"模型源码校验失败：{relative_name}")
sys.path.insert(0, str(DINO_SOURCE_DIR))
from dinov2.hub.backbones import dinov2_vitb14_reg
(WORK_DIR / "source_manifest.json").write_text(
    json.dumps(SOURCE_MANIFEST, indent=2, ensure_ascii=False), encoding="utf-8")
print("已准备原折划分、提交 ID 顺序和 DINOv2 源码。")

## 3. 数据定位与核验

保留任务原有的五类各 100 张训练图和 378 张测试图。训练文件名、标签和原折划分
必须吻合；提交模板按 ID 核验，不使用其中的 Category 值作为测试标签。

In [ ]:
from __future__ import annotations

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}


IMAGENET_MEAN = (0.485, 0.456, 0.406)


IMAGENET_STD = (0.229, 0.224, 0.225)


@dataclass(frozen=True)
class ImageRecord:
    path: Path
    label: int
    class_name: str


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def image_to_rgb(image: Image.Image) -> Image.Image:
    if image.mode == "RGBA":
        background = Image.new("RGBA", image.size, (255, 255, 255, 255))
        return Image.alpha_composite(background, image).convert("RGB")
    return image.convert("RGB")


class PlantDataset(Dataset):
    def __init__(self, records: Sequence[ImageRecord], transform: transforms.Compose):
        self.records = list(records)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, str]:
        record = self.records[index]
        with Image.open(record.path) as image:
            tensor = self.transform(image_to_rgb(image))
        return tensor, record.label, record.path.name


def discover_training_records(train_root: Path) -> tuple[list[ImageRecord], list[str]]:
    class_names = sorted(path.name for path in train_root.iterdir() if path.is_dir())
    if not class_names:
        raise RuntimeError(f"No class folders found below {train_root}")
    class_to_index = {name: index for index, name in enumerate(class_names)}
    records: list[ImageRecord] = []
    for class_name in class_names:
        class_dir = train_root / class_name
        for path in sorted(class_dir.iterdir()):
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES:
                records.append(ImageRecord(path, class_to_index[class_name], class_name))
    return records, class_names


def discover_test_records(
    test_root: Path, submission_template: Path | None
) -> list[ImageRecord]:
    if submission_template is not None:
        with submission_template.open("r", newline="", encoding="utf-8-sig") as handle:
            rows = list(csv.DictReader(handle))
        if not rows or "ID" not in rows[0]:
            raise RuntimeError("Submission template must contain an ID column.")
        names = [row["ID"] for row in rows]
    else:
        names = sorted(
            path.name
            for path in test_root.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        )

    records = []
    for name in names:
        path = test_root / name
        if not path.exists():
            raise FileNotFoundError(f"Submission ID does not exist in test folder: {path}")
        records.append(ImageRecord(path, -1, ""))
    return records


def make_loader(
    records: Sequence[ImageRecord],
    transform: transforms.Compose,
    batch_size: int,
    workers: int,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        PlantDataset(records, transform),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=workers > 0,
        generator=generator,
    )


def classification_metrics(
    labels: np.ndarray, predictions: np.ndarray, class_names: Sequence[str]
) -> dict:
    num_classes = len(class_names)
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    for target, prediction in zip(labels, predictions, strict=True):
        confusion[int(target), int(prediction)] += 1

    per_class = {}
    f1_scores = []
    for index, class_name in enumerate(class_names):
        true_positive = int(confusion[index, index])
        false_positive = int(confusion[:, index].sum() - true_positive)
        false_negative = int(confusion[index, :].sum() - true_positive)
        support = int(confusion[index, :].sum())
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        f1_scores.append(f1)
        per_class[class_name] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    return {
        "accuracy": float(np.mean(labels == predictions)),
        "macro_f1": float(np.mean(f1_scores)),
        "per_class": per_class,
        "confusion_matrix": confusion.tolist(),
    }


def save_prediction_table(
    output_path: Path,
    names: Sequence[str],
    labels: np.ndarray,
    probabilities: np.ndarray,
    class_names: Sequence[str],
) -> None:
    with output_path.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.writer(handle)
        writer.writerow(["file", "true", "predicted", *[f"p_{name}" for name in class_names]])
        predictions = probabilities.argmax(axis=1)
        for name, label, prediction, row in zip(
            names, labels, predictions, probabilities, strict=True
        ):
            writer.writerow(
                [name, class_names[int(label)], class_names[int(prediction)], *[float(x) for x in row]]
            )

def write_json(path: Path, value: dict) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_assignments(path: Path, records, class_names: list[str]) -> np.ndarray:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    if len(rows) != len(records) or len({row["file"] for row in rows}) != len(rows):
        raise ValueError("Fold assignments are incomplete or contain duplicate filenames")
    lookup = {row["file"]: row for row in rows}
    if set(lookup) != {record.path.name for record in records}:
        raise ValueError("Current training files differ from the reference five-fold split")
    result = []
    for record in records:
        row = lookup[record.path.name]
        if row["class"] != record.class_name or int(row["label"]) != record.label:
            raise ValueError(f"Label mismatch in reference split: {record.path.name}")
        result.append(int(row["fold"]))
    assignments = np.asarray(result, dtype=np.int64)
    if set(assignments) != set(range(5)):
        raise ValueError("Expected the original five folds")
    for fold in range(5):
        indices = np.flatnonzero(assignments == fold)
        labels = [records[index].label for index in indices]
        if len(indices) != 100 or any(labels.count(label) != 20 for label in range(len(class_names))):
            raise ValueError(f"Reference fold {fold} is not the expected balanced 100-image split")
    return assignments


def normalize_probs(probabilities: np.ndarray) -> np.ndarray:
    output = np.asarray(probabilities, dtype=np.float64)
    output /= output.sum(axis=1, keepdims=True)
    return output


def save_test_table(path: Path, records, class_names: list[str], probs: np.ndarray) -> None:
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(["ID", "Category", *[f"p_{name}" for name in class_names]])
        for record, row in zip(records, probs, strict=True):
            writer.writerow([record.path.name, class_names[int(row.argmax())], *map(float, row)])


def save_submission(path: Path, records, class_names: list[str], probs: np.ndarray) -> None:
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(["ID", "Category"])
        for record, row in zip(records, probs, strict=True):
            writer.writerow([record.path.name, class_names[int(row.argmax())]])

In [ ]:
def locate_data_root():
    if DATA_ROOT:
        root = Path(DATA_ROOT).expanduser().resolve()
        if not (root / "train").is_dir() or not (root / "test").is_dir():
            raise FileNotFoundError(f"DATA_ROOT 必须包含 train 和 test：{root}")
        return root

    def candidates_under(directory):
        found = []
        if directory.is_dir():
            for train_dir in directory.rglob("train"):
                if train_dir.is_dir() and (train_dir.parent / "test").is_dir():
                    labels = sorted(path.name for path in train_dir.iterdir() if path.is_dir())
                    if labels == EXPECTED_CLASSES:
                        found.append(train_dir.parent.resolve())
        return sorted(set(found))

    # 同时挂载竞赛数据和上传的数据集时，优先使用本项目已核验的 Dataset。
    search_roots = [INPUT_ROOT]
    if PREFERRED_DATASET_SLUG and INPUT_ROOT.is_dir():
        preferred = sorted(path for path in INPUT_ROOT.rglob(PREFERRED_DATASET_SLUG) if path.is_dir())
        if INPUT_ROOT.name == PREFERRED_DATASET_SLUG:
            preferred.insert(0, INPUT_ROOT)
        if preferred:
            search_roots = preferred

    candidates = sorted({path for directory in search_roots for path in candidates_under(directory)})
    if not candidates:
        extracted_roots = []
        for directory in search_roots:
            if directory.is_dir():
                for index, zip_path in enumerate(sorted(directory.rglob("task2_dataset_for_kaggle.zip"))):
                    destination = WORK_DIR / "extracted_input" / f"{zip_path.parent.name}_{index}"
                    with zipfile.ZipFile(zip_path) as archive:
                        extract_zip_safely(archive, destination)
                    extracted_roots.append(destination)
        candidates = sorted({path for directory in extracted_roots for path in candidates_under(directory)})
    if not candidates:
        raise FileNotFoundError("未找到任务数据，请检查 Notebook 的 Input 面板或设置 DATA_ROOT。")
    if len(candidates) != 1:
        raise RuntimeError("发现多个数据目录，请设置 DATA_ROOT：\n" + "\n".join(map(str, candidates)))
    return candidates[0]


RESOLVED_DATA_ROOT = locate_data_root()
TEMPLATE_PATH = Path(SUBMISSION_TEMPLATE).resolve() if SUBMISSION_TEMPLATE else EMBEDDED_TEMPLATE
TRAIN_RECORDS, CLASS_NAMES = discover_training_records(RESOLVED_DATA_ROOT / "train")
if CLASS_NAMES != EXPECTED_CLASSES:
    raise ValueError(f"类别不匹配：{CLASS_NAMES}")
counts = Counter(record.class_name for record in TRAIN_RECORDS)
if len(TRAIN_RECORDS) != 500 or any(counts[name] != 100 for name in CLASS_NAMES):
    raise ValueError(f"需要五类各 100 张训练图；实际：{dict(counts)}")
ASSIGNMENTS = read_assignments(SPLIT_PATH, TRAIN_RECORDS, CLASS_NAMES)
with TEMPLATE_PATH.open(encoding="utf-8-sig", newline="") as stream:
    template_rows = list(csv.DictReader(stream))
TEMPLATE_IDS = [row["ID"] for row in template_rows]
if len(TEMPLATE_IDS) != 378 or len(set(TEMPLATE_IDS)) != 378:
    raise ValueError("提交模板需要 378 个唯一 ID")
if set(TEMPLATE_IDS) != set(ORIGINAL_TEST_IDS) or any(Path(name).name != name for name in TEMPLATE_IDS):
    raise ValueError("提交模板 ID 与原任务不同")
actual_test_ids = {path.name for path in (RESOLVED_DATA_ROOT / "test").iterdir()
                   if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES}
if actual_test_ids != set(TEMPLATE_IDS):
    raise ValueError("测试图文件集合与提交模板不同")
TEST_RECORDS = discover_test_records(RESOLVED_DATA_ROOT / "test", TEMPLATE_PATH)
print("Data root:", RESOLVED_DATA_ROOT)
print("训练图:", len(TRAIN_RECORDS), "测试图:", len(TEST_RECORDS))
print("类别数量:", dict(counts))
print("原折划分 SHA-256:", sha256(SPLIT_PATH))
print("每折验证图:", {fold: int(np.sum(ASSIGNMENTS == fold)) for fold in range(5)})


## 4. 模型与官方预训练权重

在第 8 层提取类别 token、全体 patch 均值、软前景加权 patch 均值，
各自归一化后按 0.50 / 0.25 / 0.25 的核权重拼接成 2304 维特征。
微调仅更新第 7/8 个 Transformer block、LayerNorm 和分类头。

如果需要关闭 Internet，可把官方 `dinov2_vitb14_reg4_pretrain.pth`
作为另一个 Dataset 添加；Notebook 会自动查找，也可指定 `LOCAL_PRETRAINED`。

In [ ]:
from __future__ import annotations

def shape_descriptors(rgb, binary):
    area = binary.mean((2, 3))
    missing = area < 0.0005
    total = binary.sum((2, 3)).clamp_min(1)
    height, width = rgb.shape[-2:]
    yy = torch.linspace(-1, 1, height, device=rgb.device)[None, None, :, None]
    xx = torch.linspace(-1, 1, width, device=rgb.device)[None, None, None, :]
    mx = (binary * xx).sum((2, 3)) / total
    my = (binary * yy).sum((2, 3)) / total
    dx, dy = xx - mx[:, :, None, None], yy - my[:, :, None, None]
    vx = (binary * dx.square()).sum((2, 3)) / total
    vy = (binary * dy.square()).sum((2, 3)) / total
    cov = (binary * dx * dy).sum((2, 3)) / total
    eccentricity = ((vx - vy).square() + 4 * cov.square()).sqrt() / (vx + vy).clamp_min(1e-6)
    colors = (rgb * binary).sum((2, 3)) / total
    return torch.cat((area, mx.abs(), my.abs(), vx, vy, eccentricity, colors, missing.float()), 1)


def normalized_foreground_prior(rgb):
    rgb = rgb.float().clamp(0, 1)
    r, g, b = rgb[:, 0:1], rgb[:, 1:2], rgb[:, 2:3]
    total = (r + g + b).clamp_min(1e-6)
    fraction, rg, gb = g / total, (g - r) / total, (g - b) / total
    visible = total > 0.04
    seed = (fraction > 0.37) & (rg > 0.025) & (gb > 0.025) & visible
    candidate = (fraction > 0.355) & (rg > 0.012) & (gb > 0.015) & visible
    binary = seed.float()
    for _ in range(4):
        binary = torch.maximum(binary, F.max_pool2d(binary, 3, 1, 1) * candidate)
    shape = shape_descriptors(rgb, binary)
    weights = 0.05 + 0.95 * F.avg_pool2d(F.max_pool2d(binary, 3, 1, 1), 5, 1, 2)
    weights = torch.where(shape[:, -1, None, None, None].bool(), torch.ones_like(weights), weights)
    return weights, shape, binary

class MidlayerPlantClassifier(nn.Module):
    """Freeze blocks 0-5, optionally train blocks 6-7, and stop at layer 8."""

    def __init__(self, backbone: nn.Module, classes: int, head_dropout: float = 0.1):
        super().__init__()
        if backbone.embed_dim != 768 or backbone.num_register_tokens != 4:
            raise ValueError("Expected DINOv2 ViT-B/14 with four register tokens")
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(head_dropout), nn.Linear(3 * 768, classes))
        self.register_buffer("rgb_mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("rgb_std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1), persistent=False)
        self.set_backbone_trainable(False)

    def set_backbone_trainable(self, enabled: bool) -> None:
        for parameter in self.backbone.parameters():
            parameter.requires_grad_(False)
        if enabled:
            for block in self.backbone.blocks[6:8]:
                for parameter in block.parameters():
                    parameter.requires_grad_(True)
            for parameter in self.backbone.norm.parameters():
                parameter.requires_grad_(True)

    def train(self, mode: bool = True):
        super().train(mode)
        # The frozen prefix must keep deterministic evaluation behavior.
        for block in self.backbone.blocks[:6]:
            block.eval()
        return self

    def features(self, images: torch.Tensor) -> torch.Tensor:
        if images.shape[-2] != images.shape[-1] or images.shape[-1] % 14:
            raise ValueError("DINOv2 input must be square and divisible by 14")
        with torch.no_grad():
            tokens = self.backbone.prepare_tokens_with_masks(images)
            for block in self.backbone.blocks[:6]:
                tokens = block(tokens)
        for block in self.backbone.blocks[6:8]:
            tokens = block(tokens)
        tokens = self.backbone.norm(tokens)
        cls = tokens[:, 0].float()
        patches = tokens[:, 1 + self.backbone.num_register_tokens:].float()
        grid = images.shape[-1] // 14
        if patches.shape[1] != grid * grid:
            raise RuntimeError("Unexpected patch grid")
        with torch.no_grad():
            rgb = (images.float() * self.rgb_std + self.rgb_mean).clamp(0, 1)
            mask, _, _ = normalized_foreground_prior(rgb)
            weights = F.adaptive_avg_pool2d(mask, (grid, grid)).flatten(1)
        plant = (patches * weights[:, :, None]).sum(1) / weights.sum(1).clamp_min(1e-6)[:, None]
        pieces = (F.normalize(cls, dim=1), F.normalize(patches.mean(1), dim=1),
                  F.normalize(plant, dim=1))
        return torch.cat((pieces[0] * (0.5 ** 0.5), pieces[1] * 0.5, pieces[2] * 0.5), dim=1)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(images) * 10.0)

WEIGHTS_NAME = "dinov2_vitb14_reg4_pretrain.pth"
WEIGHTS_URL = "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_reg4_pretrain.pth"
PRETRAINED_PATH = Path(torch.hub.get_dir()) / "checkpoints" / WEIGHTS_NAME

def prepare_pretrained():
    PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
    expected_hash = SOURCE_MANIFEST["pretrained_sha256"]
    if not PRETRAINED_PATH.exists():
        candidates = ([Path(LOCAL_PRETRAINED).expanduser().resolve()] if LOCAL_PRETRAINED else
                      sorted(INPUT_ROOT.rglob(WEIGHTS_NAME)) if INPUT_ROOT.is_dir() else [])
        if candidates:
            valid = next((path for path in candidates if path.is_file() and sha256(path) == expected_hash), None)
            if valid is None:
                raise ValueError("输入的预训练权重不存在或 SHA-256 与原实验不同")
            shutil.copyfile(valid, PRETRAINED_PATH)
        else:
            if LOCAL_PRETRAINED:
                raise FileNotFoundError(LOCAL_PRETRAINED)
            temporary = PRETRAINED_PATH.with_suffix(".download")
            try:
                torch.hub.download_url_to_file(WEIGHTS_URL, str(temporary), hash_prefix=expected_hash)
                temporary.replace(PRETRAINED_PATH)
            except Exception as error:
                raise RuntimeError("权重下载失败。请开启 Internet，或添加官方权重 Dataset 并设置 LOCAL_PRETRAINED。") from error
    if sha256(PRETRAINED_PATH) != expected_hash:
        raise ValueError("预训练权重 SHA-256 与原实验不同")
    print("预训练权重:", PRETRAINED_PATH, "SHA-256:", expected_hash)

def load_model(classes: int) -> MidlayerPlantClassifier:
    # 原官方骨干的初始化方式保持一致；只改为从已核验的本地文件读权重。
    backbone = dinov2_vitb14_reg(pretrained=False)
    state = torch.load(PRETRAINED_PATH, map_location="cpu", weights_only=True)
    backbone.load_state_dict(state, strict=True)
    return MidlayerPlantClassifier(backbone, classes)

prepare_pretrained()

## 5. 固定训练流程与 SVC

保留原增强、AdamW 分层学习率、余弦调度、FP16、梯度累积、标签平滑，
以及训练折内的三折 SVC 搜索。外折验证图不用于选择训练轮数或 SVC 超参数。
SVC 内层特征由全部外折训练标签微调得到，其内层分数不能视作独立泛化成绩；
外层 OOF 用于本次评估。

重跑相同配置会复用已完成折；修改训练设置时请换一个 `RUN_NAME`。

In [ ]:
from __future__ import annotations

def gram(train_features: np.ndarray, test_features: np.ndarray, kind: str,
         gamma: float) -> tuple[np.ndarray, np.ndarray]:
    similarity = np.clip(train_features @ train_features.T, -1, 1)
    test_similarity = np.clip(test_features @ train_features.T, -1, 1)
    if kind == "linear":
        return similarity, test_similarity
    if kind == "rbf":
        return np.exp(-2 * gamma * (1 - similarity)), np.exp(-2 * gamma * (1 - test_similarity))
    raise ValueError(kind)


def choose_params(features: np.ndarray, labels: np.ndarray, outer_train: np.ndarray,
                  seed: int, c_values: list[float]) -> tuple[tuple[str, float, float], list[dict]]:
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    splits = [(outer_train[a], outer_train[b]) for a, b in inner.split(features[outer_train], labels[outer_train])]
    candidates = []
    # Cache both kernels for each choice of geometry.
    for kind, gamma in (("linear", 0.0), ("rbf", 1.0)):
        full_kernel, _ = gram(features, features, kind, gamma)
        for c in c_values:
            predictions = np.full(len(labels), -1, dtype=np.int64)
            for inner_train, inner_val in splits:
                model = SVC(kernel="precomputed", C=c, probability=True, random_state=seed)
                model.fit(full_kernel[np.ix_(inner_train, inner_train)], labels[inner_train])
                probs = model.predict_proba(full_kernel[np.ix_(inner_val, inner_train)])
                predictions[inner_val] = model.classes_[probs.argmax(1)]
            inner_indices = np.concatenate([b for _, b in splits])
            accuracy = accuracy_score(labels[inner_indices], predictions[inner_indices])
            macro_f1 = f1_score(labels[inner_indices], predictions[inner_indices], average="macro")
            candidates.append({"kind": kind, "gamma": gamma, "C": c,
                               "accuracy": float(accuracy), "macro_f1": float(macro_f1)})
    # Prefer the simpler, more regularized model on ties.
    best = max(candidates, key=lambda row: (row["accuracy"], row["macro_f1"],
                                             row["kind"] == "linear", -row["C"]))
    return (best["kind"], best["gamma"], best["C"]), candidates

def transforms_for_run(size: int):
    train = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.84, 1.0), ratio=(0.9, 1.1),
                                     interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ColorJitter(brightness=.08, contrast=.08, saturation=.08, hue=.015),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    evaluation = transforms.Compose([
        transforms.Resize(size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(size), transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train, evaluation


def optimizer_for(model, args, stage):
    if stage == "warmup":
        groups = [{"params": model.head.parameters(), "lr": args.lr_warmup}]
    else:
        groups = [
            {"params": model.backbone.blocks[6].parameters(), "lr": args.lr_backbone / 2},
            {"params": model.backbone.blocks[7].parameters(), "lr": args.lr_backbone},
            {"params": model.backbone.norm.parameters(), "lr": args.lr_backbone},
            {"params": model.head.parameters(), "lr": args.lr_head},
        ]
    return torch.optim.AdamW(groups, weight_decay=args.weight_decay)


def train_epoch(model, loader, optimizer, scaler, criterion, device, accumulation):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    count = correct = 0
    loss_sum = 0.0
    for i, (images, labels, _) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        group_size = min(accumulation, len(loader) - (i // accumulation) * accumulation)
        with torch.autocast(device_type=device.type, dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss / group_size).backward()
        count += len(labels)
        correct += int((logits.argmax(1) == labels).sum())
        loss_sum += float(loss.detach()) * len(labels)
        if (i + 1) % accumulation == 0 or i + 1 == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
    return {"loss": loss_sum / count, "train_accuracy": correct / count}


@torch.inference_mode()
def extract(model, loader, device):
    model.eval()
    features, probabilities, labels, names = [], [], [], []
    for images, batch_labels, batch_names in loader:
        images = images.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16):
            value = model.features(images)
            logits = model.head(value * 10.0)
        features.append(value.float().cpu().numpy())
        probabilities.append(logits.float().softmax(1).cpu().numpy())
        labels.extend(batch_labels.tolist())
        names.extend(batch_names)
    output = np.concatenate(features).astype(np.float64)
    if not np.isfinite(output).all() or not np.allclose(np.linalg.norm(output, axis=1), 1, atol=1e-4):
        raise RuntimeError("Extracted features are not finite, unit-length vectors")
    return output, normalize_probs(np.concatenate(probabilities)), np.asarray(labels), names


def save_checkpoint(path, model, fold, classes, config_path):
    kept = ("backbone.blocks.6.", "backbone.blocks.7.", "backbone.norm.", "head.")
    state = {key: value.detach().cpu() for key, value in model.state_dict().items()
             if key.startswith(kept)}
    torch.save({"trained_state": state, "fold": fold, "class_names": classes,
                "architecture": "DINOv2-B/14-register, layer 8, train blocks 7-8",
                "config_sha256": sha256(config_path)}, path)

def run_training(args):
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is needed for this fine-tuning run")
    if args.image_size % 14 or min(args.batch_size, args.accumulation, args.warmup_epochs,
                                   args.finetune_epochs) < 1:
        raise ValueError("Invalid image size or training schedule")
    selected_folds = sorted(set(args.fold_index or range(5)))
    if not set(selected_folds).issubset(set(range(5))):
        raise ValueError("Fold indices must be 0-4")
    c_values = [float(value) for value in args.c_grid.split(",")]
    if any(value <= 0 or not np.isfinite(value) for value in c_values):
        raise ValueError("C grid must contain positive finite values")
    data_root = args.data_root.resolve()
    output_dir = args.output_dir.resolve()
    records, classes = discover_training_records(data_root / "train")
    test_records = discover_test_records(data_root / "test", args.submission_template.resolve())
    split_path = args.reference_run.resolve() / "fold_assignments.csv"
    assignments = read_assignments(split_path, records, classes)
    weights_path = Path(torch.hub.get_dir()) / "checkpoints" / "dinov2_vitb14_reg4_pretrain.pth"
    config = {key: str(value.resolve()) if isinstance(value, Path) else value
              for key, value in vars(args).items() if key not in ("fold_index", "resume")}
    config.update({"class_names": classes, "split_sha256": sha256(split_path),
                   "pretrained_sha256": sha256(weights_path), "torch_version": torch.__version__,
                   "validation_policy": "Fixed schedule and inner SVC CV; outer labels never select checkpoints or hyperparameters."})
    config_path = output_dir / "config.json"
    if config_path.exists():
        if not args.resume or json.loads(config_path.read_text(encoding="utf-8")) != config:
            raise ValueError("Existing run has another config; use --resume or a new output directory")
    else:
        if output_dir.exists() and any(output_dir.iterdir()):
            raise FileExistsError("Output directory is nonempty")
        output_dir.mkdir(parents=True, exist_ok=True)
        write_json(config_path, config)
        shutil.copyfile(split_path, output_dir / "fold_assignments.csv")
    train_transform, eval_transform = transforms_for_run(args.image_size)
    device = torch.device("cuda")
    labels_all = np.asarray([record.label for record in records])
    for fold in selected_folds:
        start = time.perf_counter()
        pred_path = output_dir / f"fold_{fold}_predictions.npz"
        checkpoint_path = output_dir / f"fold_{fold}_final.pt"
        metrics_path = output_dir / f"fold_{fold}_metrics.json"
        svc_path = output_dir / f"fold_{fold}_svc.joblib"
        if args.resume and all(path.exists() for path in (pred_path, checkpoint_path, metrics_path, svc_path)):
            print(f"fold={fold} reused completed artifacts", flush=True)
            continue
        set_seed(args.seed + fold)
        train_indices = np.flatnonzero(assignments != fold)
        val_indices = np.flatnonzero(assignments == fold)
        train_records = [records[index] for index in train_indices]
        val_records = [records[index] for index in val_indices]
        train_loader = make_loader(train_records, train_transform, args.batch_size,
                                   args.workers, True, args.seed + fold)
        train_eval_loader = make_loader(train_records, eval_transform, args.batch_size,
                                        args.workers, False, args.seed + fold)
        val_loader = make_loader(val_records, eval_transform, args.batch_size,
                                 args.workers, False, args.seed + fold)
        test_loader = make_loader(test_records, eval_transform, args.batch_size,
                                  args.workers, False, args.seed + fold)
        model = load_model(len(classes)).to(device)
        criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
        scaler = torch.amp.GradScaler("cuda")
        history = []
        for stage, epochs in (("warmup", args.warmup_epochs), ("finetune", args.finetune_epochs)):
            model.set_backbone_trainable(stage == "finetune")
            optimizer = optimizer_for(model, args, stage)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
            for epoch in range(1, epochs + 1):
                result = train_epoch(model, train_loader, optimizer, scaler, criterion,
                                     device, args.accumulation)
                history.append({"stage": stage, "epoch": epoch, **result})
                scheduler.step()
                write_json(output_dir / f"fold_{fold}_progress.json", {"history": history})
                print(f"fold={fold} {stage} {epoch}/{epochs} loss={result['loss']:.4f} "
                      f"train_accuracy={result['train_accuracy']:.4f}", flush=True)
            del optimizer, scheduler
        train_features, train_head, train_labels, train_names = extract(model, train_eval_loader, device)
        val_features, val_head, val_labels, val_names = extract(model, val_loader, device)
        test_features, test_head, _, test_names = extract(model, test_loader, device)
        if train_names != [record.path.name for record in train_records] or \
                val_names != [record.path.name for record in val_records] or \
                test_names != [record.path.name for record in test_records]:
            raise RuntimeError("Extracted feature order differs from record order")
        if not np.array_equal(train_labels, labels_all[train_indices]) or \
                not np.array_equal(val_labels, labels_all[val_indices]):
            raise RuntimeError("Labels differ from reference data")
        all_features = np.empty((len(records), train_features.shape[1]), dtype=np.float64)
        all_features[train_indices], all_features[val_indices] = train_features, val_features
        params, search = choose_params(all_features, labels_all, train_indices,
                                       4200 + fold, c_values)
        kind, gamma, c = params
        train_kernel, val_kernel = gram(train_features, val_features, kind, gamma)
        _, test_kernel = gram(train_features, test_features, kind, gamma)
        svc = SVC(kernel="precomputed", C=c, probability=True, random_state=4200 + fold)
        svc.fit(train_kernel, train_labels)
        if not np.array_equal(svc.classes_, np.arange(len(classes))):
            raise RuntimeError("Unexpected SVC class ordering")
        val_svc = normalize_probs(svc.predict_proba(val_kernel))
        test_svc = normalize_probs(svc.predict_proba(test_kernel))
        np.savez_compressed(pred_path, val_head=val_head, test_head=test_head,
                            val_svc=val_svc, test_svc=test_svc,
                            val_files=np.asarray(val_names), test_files=np.asarray(test_names))
        joblib.dump({"model": svc, "train_features": train_features,
                     "train_files": np.asarray(train_names), "params": params,
                     "class_names": classes}, svc_path)
        save_checkpoint(checkpoint_path, model, fold, classes, config_path)
        metrics = {"head": classification_metrics(val_labels, val_head.argmax(1), classes),
                   "svc": classification_metrics(val_labels, val_svc.argmax(1), classes),
                   "mean": classification_metrics(val_labels,
                                                  (val_head + val_svc).argmax(1), classes)}
        write_json(metrics_path, {"fold": fold, "metrics": metrics,
                                  "svc_params": params, "svc_inner_search": search,
                                  "history": history, "validation_files": val_names,
                                  "elapsed_seconds": time.perf_counter() - start})
        print(f"fold={fold} head={metrics['head']['accuracy']:.3f} "
              f"svc={metrics['svc']['accuracy']:.3f} mean={metrics['mean']['accuracy']:.3f} "
              f"params={params}", flush=True)
        del model, scaler
        torch.cuda.empty_cache()
    completed = [fold for fold in range(5) if all((output_dir / f"fold_{fold}_{suffix}").exists()
                 for suffix in ("predictions.npz", "final.pt", "metrics.json", "svc.joblib"))]
    summary = {"completed_folds": completed, "evaluated_images": 100 * len(completed),
               "validation_policy": config["validation_policy"], "methods": {}}
    for method in ("head", "svc", "mean"):
        oof = np.full((len(records), len(classes)), np.nan)
        test_arrays = []
        fold_accuracies = []
        for fold in completed:
            with np.load(output_dir / f"fold_{fold}_predictions.npz", allow_pickle=False) as archive:
                indices = np.flatnonzero(assignments == fold)
                if archive["val_files"].astype(str).tolist() != [records[i].path.name for i in indices]:
                    raise RuntimeError("Saved validation IDs differ from fold")
                if archive["test_files"].astype(str).tolist() != [r.path.name for r in test_records]:
                    raise RuntimeError("Saved test IDs differ from template")
                val = (archive["val_head"] + archive["val_svc"]) / 2 if method == "mean" else archive[f"val_{method}"]
                test = (archive["test_head"] + archive["test_svc"]) / 2 if method == "mean" else archive[f"test_{method}"]
                oof[indices] = normalize_probs(val)
                test_arrays.append(normalize_probs(test))
                fold_accuracies.append(float(accuracy_score(labels_all[indices], val.argmax(1))))
        available = np.flatnonzero(np.isfinite(oof).all(1))
        result = classification_metrics(labels_all[available], oof[available].argmax(1), classes)
        result["fold_accuracy"] = fold_accuracies
        summary["methods"][method] = result
        save_prediction_table(output_dir / f"oof_{method}.csv",
                              [records[i].path.name for i in available],
                              labels_all[available], oof[available], classes)
        if len(completed) == 5:
            test_probs = normalize_probs(np.mean(test_arrays, axis=0))
            save_test_table(output_dir / f"test_probabilities_{method}.csv",
                            test_records, classes, test_probs)
            save_submission(output_dir / f"submission_{method}.csv", test_records,
                            classes, test_probs)
    write_json(output_dir / "metrics.json", summary)
    if len(completed) == 5:
        # The SVC is the experiment's deployable classifier; expose standard names.
        for source, target in (("oof_svc.csv", "oof_predictions.csv"),
                               ("test_probabilities_svc.csv", "test_probabilities.csv"),
                               ("submission_svc.csv", "submission.csv")):
            shutil.copyfile(output_dir / source, output_dir / target)
    print(json.dumps({"completed_folds": completed,
                      "accuracy": {k: v["accuracy"] for k, v in summary["methods"].items()}},
                     ensure_ascii=False), flush=True)

In [ ]:
args = SimpleNamespace(
    data_root=RESOLVED_DATA_ROOT,
    submission_template=TEMPLATE_PATH,
    reference_run=REFERENCE_DIR,
    output_dir=RUN_DIR,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, accumulation=ACCUMULATION,
    workers=WORKERS, warmup_epochs=WARMUP_EPOCHS, finetune_epochs=FINETUNE_EPOCHS,
    lr_warmup=LR_WARMUP, lr_backbone=LR_BACKBONE, lr_head=LR_HEAD,
    weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING,
    c_grid=C_GRID, seed=SEED, fold_index=FOLDS, resume=RESUME,
)
if not FOLDS or any(fold not in range(5) for fold in FOLDS):
    raise ValueError("FOLDS 必须是 0 到 4 中至少一个折号")
if len(set(FOLDS)) != len(FOLDS):
    raise ValueError("FOLDS 不能重复")
print("实验目录:", RUN_DIR)
print("本次运行折:", FOLDS)
run_training(args)

## 6. OOF、折隔离检查与提交导出

检查各 SVC 只包含相应外折的 400 张训练图、验证 ID 与原划分一致、
概率归一化、提交 ID 顺序及类别。只有五折全部完成才导出标准提交文件。

In [ ]:
def read_table(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

def checked_probabilities(rows, classes):
    probabilities = np.asarray([[float(row[f"p_{name}"]) for name in classes] for row in rows])
    if (not np.isfinite(probabilities).all() or (probabilities < 0).any()
            or not np.allclose(probabilities.sum(1), 1, atol=1e-5)):
        raise ValueError("保存的概率不合法")
    return probabilities

def validate_run(run_dir, require_complete=True):
    summary = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    completed = summary["completed_folds"]
    if require_complete and completed != list(range(5)):
        raise RuntimeError(f"完整提交需要五折；当前完成：{completed}")
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    if config["class_names"] != CLASS_NAMES or config["split_sha256"] != sha256(SPLIT_PATH):
        raise ValueError("运行配置的类别或折划分不一致")
    if sha256(run_dir / "fold_assignments.csv") != sha256(SPLIT_PATH):
        raise ValueError("运行目录内折划分不一致")
    test_arrays = []
    reconstructed_oof = np.full((len(TRAIN_RECORDS), len(CLASS_NAMES)), np.nan)
    expected_oof = []
    for fold in completed:
        train_names = [r.path.name for r, owner in zip(TRAIN_RECORDS, ASSIGNMENTS) if owner != fold]
        val_names = [r.path.name for r, owner in zip(TRAIN_RECORDS, ASSIGNMENTS) if owner == fold]
        artifact = joblib.load(run_dir / f"fold_{fold}_svc.joblib")
        if artifact["train_files"].astype(str).tolist() != train_names:
            raise ValueError(f"第 {fold} 折 SVC 训练样本与原划分不同")
        if artifact["class_names"] != CLASS_NAMES or not np.array_equal(artifact["model"].classes_, np.arange(len(CLASS_NAMES))):
            raise ValueError(f"第 {fold} 折 SVC 类别顺序不同")
        checkpoint = torch.load(run_dir / f"fold_{fold}_final.pt", map_location="cpu", weights_only=True)
        if (checkpoint["fold"] != fold or checkpoint["class_names"] != CLASS_NAMES
                or checkpoint["config_sha256"] != sha256(run_dir / "config.json")):
            raise ValueError(f"第 {fold} 折权重元数据不一致")
        del checkpoint
        with np.load(run_dir / f"fold_{fold}_predictions.npz", allow_pickle=False) as archive:
            if archive["val_files"].astype(str).tolist() != val_names:
                raise ValueError(f"第 {fold} 折验证 ID 不一致")
            if archive["test_files"].astype(str).tolist() != TEMPLATE_IDS:
                raise ValueError(f"第 {fold} 折测试 ID 顺序不一致")
            for name in ("val_svc", "test_svc"):
                probs = archive[name]
                if not np.isfinite(probs).all() or (probs < 0).any() or not np.allclose(probs.sum(1), 1, atol=1e-5):
                    raise ValueError(f"第 {fold} 折 {name} 概率不合法")
            reconstructed_oof[ASSIGNMENTS == fold] = archive["val_svc"]
            test_arrays.append(archive["test_svc"])
        expected_oof.extend(val_names)
    oof_rows = read_table(run_dir / "oof_svc.csv")
    if len(oof_rows) != len(expected_oof) or {row["file"] for row in oof_rows} != set(expected_oof):
        raise ValueError("OOF 图像数或文件集合不一致")
    available = np.flatnonzero(np.isfinite(reconstructed_oof).all(1))
    records = [TRAIN_RECORDS[index] for index in available]
    if [row["file"] for row in oof_rows] != [record.path.name for record in records]:
        raise ValueError("OOF 图像顺序不同")
    if [row["true"] for row in oof_rows] != [record.class_name for record in records]:
        raise ValueError("OOF 真值类别不同")
    oof_probs = checked_probabilities(oof_rows, CLASS_NAMES)
    if not np.allclose(oof_probs, reconstructed_oof[available], atol=1e-12):
        raise ValueError("OOF 概率与逐折文件不同")
    truth = np.asarray([record.label for record in records])
    recalculated = classification_metrics(truth, oof_probs.argmax(1), CLASS_NAMES)
    for metric in ("accuracy", "macro_f1"):
        if not np.isclose(recalculated[metric], summary["methods"]["svc"][metric]):
            raise ValueError(f"{metric} 与保存结果不一致")
    audit = {"completed_folds": completed, "fold_isolation_verified": True,
             "oof_images": len(oof_rows), "oof_correct": int(np.sum(truth == oof_probs.argmax(1))),
             "accuracy": recalculated["accuracy"], "macro_f1": recalculated["macro_f1"]}
    if len(completed) == 5:
        submission_rows = read_table(run_dir / "submission.csv")
        test_rows = read_table(run_dir / "test_probabilities.csv")
        if any([row["ID"] for row in rows] != TEMPLATE_IDS for rows in (submission_rows, test_rows)):
            raise ValueError("提交 ID 数量或顺序与模板不一致")
        probabilities = checked_probabilities(test_rows, CLASS_NAMES)
        expected = normalize_probs(np.mean(test_arrays, axis=0))
        if not np.allclose(probabilities, expected, atol=1e-12):
            raise ValueError("测试概率不等于五折均值")
        predicted_categories = [CLASS_NAMES[int(index)] for index in probabilities.argmax(1)]
        if any([row["Category"] for row in rows] != predicted_categories for rows in (submission_rows, test_rows)):
            raise ValueError("提交类别与测试概率不一致")
        for canonical, specialized in (("submission.csv", "submission_svc.csv"),
                                      ("test_probabilities.csv", "test_probabilities_svc.csv"),
                                      ("oof_predictions.csv", "oof_svc.csv")):
            if (run_dir / canonical).read_bytes() != (run_dir / specialized).read_bytes():
                raise ValueError(f"{canonical} 与 SVC 输出不一致")
        audit.update(test_images=len(submission_rows), submission_valid=True)
    write_json(run_dir / "notebook_audit.json", audit)
    return audit

SUMMARY = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
AUDIT = validate_run(RUN_DIR, require_complete=False)
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))
print("SVC 各折 Accuracy:", SUMMARY["methods"]["svc"]["fold_accuracy"])
if SUMMARY["completed_folds"] == list(range(5)):
    SUBMISSION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    if SUBMISSION_OUTPUT.resolve() != (RUN_DIR / "submission.csv").resolve():
        shutil.copyfile(RUN_DIR / "submission.csv", SUBMISSION_OUTPUT)
    print("提交文件:", SUBMISSION_OUTPUT)
    print("类别分布:", dict(Counter(row["Category"] for row in read_table(SUBMISSION_OUTPUT))))
    display(FileLink(str(SUBMISSION_OUTPUT)))
    display(FileLink(str(RUN_DIR / "metrics.json")))
else:
    print("当前为部分折结果。将 FOLDS 设置为 [0, 1, 2, 3, 4]，保持配置不变并重跑训练与导出单元格。")

## 7. 可选：重新加载五折权重核验推理

设置 `VERIFY_RELOADED=True` 会独立加载保存的权重和 SVC，重新推理 378 张测试图，
检查概率和类别与训练流程的输出一致。默认关闭，以免重复进行五折推理。
完整产物保存在 `plant_midlayer/experiments/<RUN_NAME>/`，包含微调权重、SVC、
OOF、逐折指标及配置。下载后应同时保留官方预训练权重和本 Notebook，
因为每折 `.pt` 只保存可训练的 block、LayerNorm 和分类头。

In [ ]:
from __future__ import annotations

def predict_records(records, run_dir: Path, batch_size: int, device: torch.device):
    config_path = run_dir / "config.json"
    config = json.loads(config_path.read_text(encoding="utf-8"))
    classes = config["class_names"]
    weights_path = Path(torch.hub.get_dir()) / "checkpoints" / "dinov2_vitb14_reg4_pretrain.pth"
    if sha256(weights_path) != config["pretrained_sha256"]:
        raise ValueError("Pretrained DINOv2 checkpoint does not match the experiment")
    _, eval_transform = transforms_for_run(int(config["image_size"]))
    loader = make_loader(records, eval_transform, batch_size, 0, False, int(config["seed"]))
    result = np.zeros((len(records), len(classes)), dtype=np.float64)
    expected_names = [record.path.name for record in records]
    for fold in range(5):
        checkpoint = torch.load(run_dir / f"fold_{fold}_final.pt", map_location="cpu",
                                weights_only=False)
        if checkpoint["fold"] != fold or checkpoint["class_names"] != classes or \
                checkpoint["config_sha256"] != sha256(config_path):
            raise ValueError(f"Fold {fold} checkpoint metadata mismatch")
        model = load_model(len(classes))
        missing, unexpected = model.load_state_dict(checkpoint["trained_state"], strict=False)
        train_prefixes = ("backbone.blocks.6.", "backbone.blocks.7.",
                          "backbone.norm.", "head.")
        if unexpected or any(name.startswith(train_prefixes) for name in missing):
            raise RuntimeError(f"Fold {fold} checkpoint is incomplete")
        model.eval().to(device)
        features, _, _, names = extract(model, loader, device)
        if names != expected_names:
            raise RuntimeError("Prediction IDs differ from input order")
        artifact = joblib.load(run_dir / f"fold_{fold}_svc.joblib")
        if artifact["class_names"] != classes or \
                not np.array_equal(artifact["model"].classes_, np.arange(len(classes))):
            raise ValueError("SVC class order mismatch")
        kind, gamma, _ = artifact["params"]
        _, kernel = gram(np.asarray(artifact["train_features"], dtype=np.float64),
                         features, kind, gamma)
        result += normalize_probs(artifact["model"].predict_proba(kernel)) / 5
        print(f"predicted fold={fold} images={len(records)}", flush=True)
        del model
        torch.cuda.empty_cache()
    if not np.isfinite(result).all() or not np.allclose(result.sum(1), 1, atol=1e-5):
        raise RuntimeError("Invalid ensemble probabilities")
    return normalize_probs(result), classes

if VERIFY_RELOADED:
    if SUMMARY["completed_folds"] != list(range(5)):
        raise RuntimeError("重新加载核验需要五折完整产物")
    reloaded, reloaded_classes = predict_records(TEST_RECORDS, RUN_DIR, BATCH_SIZE, torch.device("cuda"))
    saved = checked_probabilities(read_table(RUN_DIR / "test_probabilities.csv"), CLASS_NAMES)
    max_difference = float(np.max(np.abs(reloaded - saved)))
    changed = int(np.count_nonzero(reloaded.argmax(1) != saved.argmax(1)))
    if reloaded_classes != CLASS_NAMES or max_difference > 1e-5 or changed:
        raise RuntimeError(f"重载核验失败：最大概率差={max_difference}, 改判={changed}")
    save_test_table(RUN_DIR / "direct_test_probabilities_svc.csv", TEST_RECORDS, CLASS_NAMES, reloaded)
    verification = {"max_abs_probability_difference": max_difference, "changed_categories": changed}
    write_json(RUN_DIR / "reload_verification.json", verification)
    print(json.dumps(verification, indent=2))
else:
    print("可选重载核验未开启；逐折数据隔离和提交格式检查已在上一节完成。")